# 165 — Alucinación, grounding y abstención

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** En un sistema con grounding, la lista `evidence` es la traza de las fuentes y
hechos que soportan la respuesta: permite verificar la atribución (que cada afirmación esté
respaldada) y auditar si hubo alucinación extrínseca.


In [ ]:
result = run_lab("evaluation", seed=165)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
umbral 0.0 (8 respondidas): correctas 5 -> cobertura 8/8=1.00, riesgo 3/8=0.375
umbral 0.8 (>=0.8: 0.95,0.9,0.85,0.8 -> 4): errores {0.85} -> cobertura 4/8=0.50, riesgo 1/4=0.25
umbral 0.9 (>=0.9: 0.95,0.9 -> 2): errores 0 -> cobertura 2/8=0.25, riesgo 0/2=0.00
```

Para un dominio legal (error caro) elegiría un umbral alto (0.9): riesgo 0 entre lo respondido a
costa de derivar el 75 % a revisión humana. La decisión equilibra costo del error vs carga de
derivación.

**Ejercicio 3.**

```text
(a) intrínseca  (contradice el documento fuente)
(b) extrínseca  (añade dato no presente en ninguna fuente)
(c) intrínseca/de fidelidad (tergiversa el contenido de la fuente: cita un artículo inexistente en ella)
```

**Ejercicio 4 (esquema).** Umbral alto de confianza calibrada; exigir que cada dosis cite la ficha
técnica del fármaco y descartar respuestas no atribuibles; derivar a un profesional ante baja
confianza, interacción no cubierta por las fuentes o dosis pediátrica/renal fuera de rango.
Métricas a reguladores: tasa de alucinación, riesgo selectivo, cobertura y % attributable-to-source,
por subgrupo de fármacos.


In [ ]:
# Verificación numérica del Ejercicio 2
preds = [(0.95,1),(0.9,1),(0.85,0),(0.8,1),(0.7,1),(0.65,0),(0.6,0),(0.55,1)]
def cov_risk(preds, thr):
    resp = [a for c, a in preds if c >= thr]
    cov = len(resp) / len(preds)
    risk = (len(resp) - sum(resp)) / len(resp) if resp else 0.0
    return cov, risk
for thr in (0.0, 0.8, 0.9):
    cov, risk = cov_risk(preds, thr)
    print(f"umbral {thr}: cobertura={cov:.2f} riesgo={risk:.3f}")
assert cov_risk(preds, 0.9) == (0.25, 0.0)


## Reflexión (guía)

1. Porque surge del objetivo generativo (premiar continuaciones plausibles); se mitiga con
   grounding, abstención y verificación, pero no se elimina con un ajuste puntual.
2. La abstención por umbral solo reduce el riesgo si la confianza refleja la probabilidad real de
   acierto; sin calibración, el umbral deja pasar errores con confianza inflada.
3. Los modelos generan citas plausibles pero pueden ser inexistentes o no soportar la afirmación;
   hay que verificar la atribución, no confiar en que exista una cita.
